# 对预测文件进行后处理

## 1.对预测文件进行提取最大连通域

In [1]:
import numpy as np
import nibabel as nib
from scipy.ndimage import label

# 1. 加载nii.gz文件
img = nib.load('18757.nii.gz')  # 请将 'your_file.nii.gz' 替换为您的文件名
data = img.get_fdata()

# 2. 将元素1和2都修改为1
data[(data == 1) | (data == 2)] = 1

# 3. 提取元素1的连通域
# 定义连通结构，这里使用26连通
# structure = np.ones((3, 3, 3), dtype=np.int8) # 26
# structure = np.array([
#     [[0, 1, 0], 
#      [1, 1, 1], 
#      [0, 1, 0]],

#     [[1, 1, 1], 
#      [1, 1, 1], 
#      [1, 1, 1]],

#     [[0, 1, 0], 
#      [1, 1, 1], 
#      [0, 1, 0]]
# ], dtype=np.int8) # 18
structure = np.array([[[0, 0, 0], 
                       [0, 1, 0], 
                       [0, 0, 0]],

                      [[0, 1, 0], 
                       [1, 1, 1], 
                       [0, 1, 0]],

                      [[0, 0, 0], 
                       [0, 1, 0], 
                       [0, 0, 0]]], dtype=np.int8) # 6
labeled_array, num_features = label(data == 1, structure=structure)

# 4. 计算每个连通域的体积并排序
volumes = []
for i in range(1, num_features + 1):
    volume = np.sum(labeled_array == i)
    volumes.append((i, volume))

# 按体积大小降序排序
volumes.sort(key=lambda x: x[1], reverse=True)

# 5. 选择体积最大的两个部分并赋值
new_data = np.zeros_like(data, dtype=np.int32)  # 将数据类型设为整型，避免浮点误差
if len(volumes) >= 2:
    largest_label = volumes[0][0]
    second_largest_label = volumes[1][0]

    # 将最大的连通域赋值为2
    new_data[labeled_array == largest_label] = 2

    # 将第二大的连通域赋值为1
    new_data[labeled_array == second_largest_label] = 1
else:
    print("连通域数量不足两个。")

# 6. 保存为新的nii.gz文件
new_img = nib.Nifti1Image(new_data, img.affine, img.header)
nib.save(new_img, '18757_pred.nii.gz')

# 可选：输出元素分布统计
unique_elements, counts = np.unique(new_data, return_counts=True)
element_distribution = dict(zip(unique_elements, counts))
print("修改后元素分布：")
for element, count in element_distribution.items():
    print(f"元素 {element}: {count} 个")


修改后元素分布：
元素 0: 194803045 个
元素 1: 2656108 个
元素 2: 3102747 个


In [6]:
import numpy as np
import nibabel as nib
from scipy.ndimage import label

# 1. 加载nii.gz文件
img = nib.load('modified_file.nii.gz')  # 请将 'your_file.nii.gz' 替换为您的文件名
data = img.get_fdata()
unique_elements, counts = np.unique(data, return_counts=True)
element_distribution = dict(zip(unique_elements, counts))
print("修改后元素分布：")
for element, count in element_distribution.items():
    print(f"元素 {element}: {count} 个")

修改后元素分布：
元素 0.0: 42604828 个
元素 1.0: 982917 个
元素 2.0: 1212255 个


## 2.对牙齿和下颌骨进行合并

In [3]:
import nibabel as nib
import numpy as np

# 加载牙颌骨和牙齿的nii.gz文件
jaw_img = nib.load('submit/18757_pred.nii.gz')
teeth_img = nib.load('submit/18757_teeth_base_yolo.nii.gz')

# 提取数据
jaw_data = jaw_img.get_fdata()
teeth_data = teeth_img.get_fdata()

# 调整牙齿标签，将1-27标签变为3-29
teeth_data = np.where(teeth_data > 0, teeth_data + 2, 0)

# 创建合并后的数据数组
combined_data = teeth_data.copy()  # 从牙齿文件开始

# 将牙颌骨数据合并到牙齿数据中
# 合并规则：只有当牙齿文件的标签为0（即背景）时才使用牙颌骨的标签
combined_data[(combined_data == 0) & (jaw_data == 1)] = 1  # 上颌骨
combined_data[(combined_data == 0) & (jaw_data == 2)] = 2  # 下颌骨

# 保存合并后的结果为新的nii.gz文件
combined_img = nib.Nifti1Image(combined_data, teeth_img.affine, teeth_img.header)
nib.save(combined_img, 'submit/18757_base_yolo_combine.nii.gz')

# 打印合并结果的元素分布，便于检查
unique_elements, counts = np.unique(combined_data, return_counts=True)
element_distribution = dict(zip(unique_elements, counts))
print("合并后元素分布：")
for element, count in element_distribution.items():
    print(f"元素 {element}: {count} 个")


合并后元素分布：
元素 0.0: 194314508 个
元素 1.0: 2559311 个
元素 2.0: 2951443 个
元素 3.0: 26273 个
元素 5.0: 25197 个
元素 9.0: 12322 个
元素 10.0: 20508 个
元素 11.0: 11089 个
元素 12.0: 43318 个
元素 13.0: 10615 个
元素 14.0: 21280 个
元素 19.0: 2837 个
元素 21.0: 19867 个
元素 27.0: 3999 个
元素 32.0: 5924 个
元素 36.0: 34331 个
元素 39.0: 34093 个
元素 42.0: 36293 个
元素 45.0: 27004 个
元素 46.0: 7336 个
元素 49.0: 31785 个
元素 50.0: 17257 个
元素 52.0: 30593 个
元素 53.0: 24241 个
元素 55.0: 12987 个
元素 56.0: 26167 个
元素 57.0: 22376 个
元素 58.0: 21714 个
元素 61.0: 37017 个
元素 63.0: 21886 个
元素 65.0: 38579 个
元素 68.0: 26859 个
元素 69.0: 31777 个
元素 79.0: 21197 个
元素 83.0: 16523 个
元素 85.0: 13394 个


In [1]:
import numpy as np
import nibabel as nib
from scipy.ndimage import label

# 1. 加载nii.gz文件
img = nib.load('1000813648_20180116_.nii.gz')  # 请将 'your_file.nii.gz' 替换为您的文件名
data = img.get_fdata()

# 2. 将元素1和2都修改为1
structure = np.array([[[0, 0, 0], 
                       [0, 1, 0], 
                       [0, 0, 0]],

                      [[0, 1, 0], 
                       [1, 1, 1], 
                       [0, 1, 0]],

                      [[0, 0, 0], 
                       [0, 1, 0], 
                       [0, 0, 0]]], dtype=np.int8) # 6
labeled_array, num_features = label(data == 1, structure=structure)

# 4. 计算每个连通域的体积并排序
volumes = []
for i in range(1, num_features + 1):
    volume = np.sum(labeled_array == i)
    volumes.append((i, volume))

# 按体积大小降序排序
volumes.sort(key=lambda x: x[1], reverse=True)

# 5. 选择体积最大的两个部分并赋值
new_data = np.zeros_like(data, dtype=np.int32)  # 将数据类型设为整型，避免浮点误差
if len(volumes) >= 2:
    largest_label = volumes[0][0]
    second_largest_label = volumes[1][0]

    # 将最大的连通域赋值为2
    new_data[labeled_array == largest_label] = 2

    # 将第二大的连通域赋值为1
    new_data[labeled_array == second_largest_label] = 1
else:
    print("连通域数量不足两个。")

# 6. 保存为新的nii.gz文件
new_img = nib.Nifti1Image(new_data, img.affine, img.header)
nib.save(new_img, '1000813648_20180116_pred.nii.gz')

# 可选：输出元素分布统计
unique_elements, counts = np.unique(new_data, return_counts=True)
element_distribution = dict(zip(unique_elements, counts))
print("修改后元素分布：")
for element, count in element_distribution.items():
    print(f"元素 {element}: {count} 个")


修改后元素分布：
元素 0: 44573276 个
元素 1: 83 个
元素 2: 226641 个
